<a href="https://colab.research.google.com/github/venkat-vipul/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/venkat-vipul/flyrank_ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb pandas pyarrow huggingface_hub fsspec

In [2]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Hugging Face login successful")

Hugging Face login successful


In [3]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("DuckDB authenticated")

DuckDB authenticated


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: One row represents the daily performance metrics of a single content page for one client on one report date. Each row contains search performance signals (such as impressions, clicks, CTR, and average position) measured for that page on that day.

Time window: The sample dataset covers June 1, 2026 through June 30, 2026 (30 days). This sample represents one month of daily observations and is used to verify the data contract before working with the full warehouse.

In [4]:
query = """
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS unique_days
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
)
"""

con.execute(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,first_date,last_date,unique_days
0,11694072,2026-06-01,2026-06-30,30


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- gsc_impressions
- gsc_clicks
- gsc_ctr
- avg_position
- content_age_days

These fields are available before making a content review decision and can be used as model inputs.

### Label / Proxy
- trend_direction

This serves as the target proxy indicating whether a page is currently declining.

### Context
- report_date
- client_hash_id
- content_hash_id

These fields identify the observation and provide context but are not used directly as predictive features.

### Excluded
- Client names or URLs (not included in the anonymized dataset)
- Future performance information
- Any field derived from the prediction target

These are excluded to prevent data leakage and improve generalization.

In [5]:
query = """
DESCRIBE
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
)
"""

con.execute(query).df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 – Verify the data grain

The intended unit of analysis is one row representing the daily performance of a single content page for one client on one report date.

The query below checks whether any duplicate combinations of report_date, client_hash_id, and content_hash_id exist. The sample dataset contains a small number of duplicate rows, but these records are exact duplicates rather than different observations.

In [6]:
query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 10
"""

con.execute(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-13,client_1a730cb2640a1abf,content_1a0c3a8cc6bdd5bc,2
1,2026-06-13,client_1a730cb2640a1abf,content_8d5a5293688d7d8e,2
2,2026-06-13,client_1a730cb2640a1abf,content_b40e27e07768f907,2
3,2026-06-13,client_1a730cb2640a1abf,content_965b9031838c130f,2
4,2026-06-13,client_1a730cb2640a1abf,content_fa351551b6d49870,2
5,2026-06-13,client_1a730cb2640a1abf,content_6604767cde89152e,2
6,2026-06-13,client_1a730cb2640a1abf,content_f58b50fc3b086cc2,2
7,2026-06-13,client_1a730cb2640a1abf,content_aaacc6b4d641648b,2
8,2026-06-13,client_1a730cb2640a1abf,content_42239c7e1161beee,2
9,2026-06-13,client_1a730cb2640a1abf,content_0e6cbd7d714dda07,2


### Query 2 – Verify dataset size and time window

This query measures the size of the sample dataset and confirms the date range covered by the data.

In [7]:
query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS unique_days
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
)
"""

con.execute(query).df()

,total_rows,first_date,last_date,unique_days
0,11694072,2026-06-01,2026-06-30,30


### Query 3 – Verify data availability

This query checks how many rows have Search Console data available. Only rows where `gsc_data_available IS TRUE` are considered available for building features from Search Console.

In [8]:
query = """
SELECT
    COUNT(*) AS available_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
)
WHERE gsc_data_available IS TRUE
"""

con.execute(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows,first_date,last_date
0,3878937,2026-06-01,2026-06-30


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limitations

This dataset is anonymized, so it cannot identify the actual client, website, or content URL behind each record. The sample dataset only covers June 2026, which limits analysis across longer time periods. Additionally, not every row has Search Console or Google Analytics data available, so some observations cannot be used when building features that depend on those sources.

In [9]:
query = """
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
)
"""

con.execute(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available,ga4_available
0,11694072,3878937.0,644726.0


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.